# ECL Quickstart Tutorial

This notebook walks through the core ECL workflow:
1. Create a synthetic model with controllable context decay
2. Compute the binned influence profile
3. Estimate ECL, ECP, AECP, and ECD
4. Visualize the results

For real genomic models, replace `SyntheticModel` with one of the model wrappers (e.g., `EnformerWrapper`).

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import matplotlib.pyplot as plt

from ecl.models.base import SyntheticModel
from ecl.influence import compute_influence_profile
from ecl.ecl import ECL, ECP, AECP, ECD, cumulative_influence, normalized_influence
from ecl.perturbations import RandomSubstitution

## 1. Create a Synthetic Model and Generate Sequences

The `SyntheticModel` has exponentially decaying influence: $w(d) = \exp(-d / \ell)$ where $\ell$ is the `decay_length`. We set $\ell = 50$ bp, meaning influence drops to $1/e$ at 50 bp from the reference.

In [ ]:
model = SyntheticModel(seq_length=500, embed_dim=64, decay_length=50.0)

rng = np.random.default_rng(42)
n_sequences = 30
sequences = rng.integers(0, 4, size=(n_sequences, 500)).astype(np.int8)
reference = 250  # center position

print(f"Model: {model.nominal_context} bp input, {model.embedding_dim}-dim embedding")
print(f"Sequences: {n_sequences} x {model.nominal_context} bp")

## 2. Compute the Influence Profile

Algorithm 2 estimates the binned influence $\hat{I}(d; r)$ for each distance $d$ from the reference.

In [ ]:
distances, influence = compute_influence_profile(
    model_fn=model,
    sequences=sequences,
    reference=reference,
    max_distance=200,
    positions_per_distance=5,
    perturbation=RandomSubstitution(),
    rng=rng,
    show_progress=True,
)

print(f"Profile shape: {influence.shape}")
print(f"Peak influence at d={distances[influence.argmax()]} bp")

## 3. Compute ECL Quantities

- **ECL** (Definition 4.1): smallest radius capturing $\beta$ fraction of total influence
- **ECP** (Definition 4.2): the curve $\beta \mapsto \text{ECL}_\beta$
- **AECP** (Definition 4.3): mean influence distance
- **ECD** (Definition 4.5): effective number of contributing positions

In [ ]:
for beta in [0.5, 0.8, 0.9, 0.95]:
    print(f"ECL_{beta:.2f} = {ECL(distances, influence, beta=beta)} bp")

betas, ecl_values = ECP(distances, influence)
aecp = AECP(distances, influence)
ecd = ECD(distances, influence)

print(f"\nAECP = {aecp:.1f} bp (mean influence distance)")
print(f"ECD  = {ecd:.1f} (effective number of positions)")

## 4. Visualize Results

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

# A: Influence profile (log scale)
ax = axes[0, 0]
ax.semilogy(distances[1:], influence[1:], "b-", linewidth=1.5)
ax.set_xlabel("Distance from reference (bp)")
ax.set_ylabel("Influence energy I(d; r)")
ax.set_title("A. Influence Profile")
ax.grid(True, alpha=0.3)

# B: Cumulative influence with ECL markers
ax = axes[0, 1]
_, cumul = cumulative_influence(distances, influence)
total = cumul[-1]
ax.plot(distances, cumul / total, "b-", linewidth=1.5)
for b in [0.5, 0.9, 0.95]:
    ecl_val = ECL(distances, influence, beta=b)
    ax.axhline(b, color="gray", ls=":", alpha=0.5)
    ax.axvline(ecl_val, color="red", ls="--", alpha=0.4)
    ax.annotate(f"ECL_{b}", (ecl_val + 2, b - 0.03), fontsize=8)
ax.set_xlabel("Radius l (bp)")
ax.set_ylabel("Cumulative influence fraction")
ax.set_title("B. Cumulative Influence & ECL")
ax.grid(True, alpha=0.3)

# C: Effective Context Profile
ax = axes[1, 0]
ax.plot(betas, ecl_values, "b-", linewidth=1.5)
ax.fill_between(betas, ecl_values, alpha=0.15)
ax.set_xlabel("Beta")
ax.set_ylabel("ECL_beta (bp)")
ax.set_title(f"C. ECP (AECP = {aecp:.0f} bp)")
ax.grid(True, alpha=0.3)

# D: Normalized influence
ax = axes[1, 1]
norm_infl = normalized_influence(distances, influence)
ax.bar(distances[:100], norm_infl[:100], width=1.0, alpha=0.7, color="steelblue")
ax.set_xlabel("Distance from reference (bp)")
ax.set_ylabel("Normalized influence")
ax.set_title(f"D. Normalized Influence (ECD = {ecd:.0f})")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()